# VADER vs RoBERTa on Amazon reviews

I compare two sentiment models on the Amazon Fine Food Reviews dataset. VADER is a rule based model that comes with NLTK. RoBERTa is a transformer model from Hugging Face. The star rating is the answer key, and I check how often each model gets it right and how long it takes per review.

This notebook runs online.

- **Kaggle** needs the Amazon Fine Food Reviews dataset added with *Add Input*, and *Internet* turned on in the settings so RoBERTa can download.
- **Colab** needs nothing extra. The notebook downloads the dataset into the Colab session by itself.

## 1. Setup

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import nltk
import pandas as pd
import seaborn as sns
import torch
from nltk.sentiment import SentimentIntensityAnalyzer
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

plt.style.use("ggplot")
pd.set_option("display.max_colwidth", 200)

LABELS = ["negative", "neutral", "positive"]
PER_LABEL = 100  # reviews per label in the test set
SEED = 42        # same seed means the same reviews on every run

FIG_DIR = Path("results/figures")
PRED_DIR = Path("results/predictions")
FIG_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)

## 2. Find the data

On Kaggle the file is already attached under `/kaggle/input`. On Colab it gets downloaded into the Colab session.

In [ ]:
def find_reviews_csv():
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        matches = sorted(kaggle_input.rglob("Reviews.csv"))
        if matches:
            return matches[0]
        raise FileNotFoundError("On Kaggle, add the Amazon Fine Food Reviews dataset with Add Input first.")

    csv_path = Path("data/Reviews.csv")
    if not csv_path.exists():
        import zipfile
        import requests

        url = "https://www.kaggle.com/api/v1/datasets/download/snap/amazon-fine-food-reviews"
        zip_path = Path("reviews.zip")
        print("Downloading the dataset into this online session (about 250 MB)...")
        with requests.get(url, stream=True, timeout=120) as response:
            response.raise_for_status()
            with open(zip_path, "wb") as f:
                for chunk in response.iter_content(chunk_size=1 << 20):
                    f.write(chunk)
        with zipfile.ZipFile(zip_path) as z:
            z.extract("Reviews.csv", "data")
        zip_path.unlink()
    return csv_path


csv_path = find_reviews_csv()
print(csv_path)

## 3. Load and clean

In [ ]:
df = pd.read_csv(csv_path, usecols=["Id", "Score", "Text"])
print(f"Rows before cleaning: {len(df):,}")

# The same review often shows up under several product variants
df = df.dropna().drop_duplicates(subset="Text")
# Some reviews contain HTML line breaks
df["Text"] = df["Text"].str.replace(r"<br\s*/?>", " ", regex=True).str.strip()

STAR_TO_LABEL = {1: "negative", 2: "negative", 3: "neutral", 4: "positive", 5: "positive"}
df["label"] = df["Score"].map(STAR_TO_LABEL)

print(f"Rows after cleaning: {len(df):,}")
df.head()

## 4. How the stars are spread out

Most reviews are 5 stars. That is why the test set takes the same number of reviews from each label. Otherwise a model could score well by saying "positive" every time.

In [ ]:
ax = df["Score"].value_counts().sort_index().plot(
    kind="bar", title="Reviews by star rating", figsize=(8, 4)
)
ax.set_xlabel("Stars")
ax.set_ylabel("Reviews")
plt.tight_layout()
plt.savefig(FIG_DIR / "stars_count.png", dpi=150)
plt.show()

df["label"].value_counts()

## 5. Build the test set

In [ ]:
sample = (
    df.groupby("label")
    .sample(n=PER_LABEL, random_state=SEED)
    .reset_index(drop=True)
)
sample["label"].value_counts()

## 6. VADER

VADER looks up each word in a list of scored words and adds them up, with a few rules for things like "not", "very", and "!!!". The compound score goes from -1 (very negative) to 1 (very positive). I use the standard cutoffs of 0.05 and -0.05.

In [ ]:
nltk.download("vader_lexicon", quiet=True)
sia = SentimentIntensityAnalyzer()


def vader_label(compound):
    if compound >= 0.05:
        return "positive"
    if compound <= -0.05:
        return "negative"
    return "neutral"


print(sia.polarity_scores("I love this coffee!"))
print(sia.polarity_scores("This is the worst coffee ever."))

In [ ]:
vader_rows = []
for _, row in tqdm(sample.iterrows(), total=len(sample), desc="VADER"):
    start = time.perf_counter()
    scores = sia.polarity_scores(row["Text"])
    seconds = time.perf_counter() - start
    vader_rows.append({
        "Id": row["Id"],
        "vader_compound": scores["compound"],
        "vader_label": vader_label(scores["compound"]),
        "vader_seconds": seconds,
    })

vader_df = pd.DataFrame(vader_rows)
vader_df.head()

## 7. RoBERTa

This model learned from about 58 million tweets. It reads the whole review at once, so it can pick up context that VADER misses. It can read at most 512 tokens, so longer reviews get cut off there.

In [ ]:
MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)
model.eval()
print("Running on", device)


def roberta_probs(text):
    """Return the chance that the text is negative, neutral, positive."""
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits[0]
    return torch.softmax(logits, dim=-1).cpu().numpy()


# The first call is slow while things warm up, so do it before timing anything
roberta_probs("warm up")
print(dict(zip(LABELS, roberta_probs("I love this coffee!").round(3))))

In [ ]:
roberta_rows = []
for _, row in tqdm(sample.iterrows(), total=len(sample), desc="RoBERTa"):
    start = time.perf_counter()
    probs = roberta_probs(row["Text"])
    seconds = time.perf_counter() - start
    roberta_rows.append({
        "Id": row["Id"],
        "roberta_neg": probs[0],
        "roberta_neu": probs[1],
        "roberta_pos": probs[2],
        "roberta_label": LABELS[int(probs.argmax())],
        "roberta_seconds": seconds,
    })

roberta_df = pd.DataFrame(roberta_rows)
roberta_df.head()

## 8. Scores

Both models are free, so cost is 0 for now. The column is there for when the LLMs join.

In [ ]:
results = sample.merge(vader_df, on="Id").merge(roberta_df, on="Id")
MODELS = [("VADER", "vader"), ("RoBERTa", "roberta")]

summary = pd.DataFrame([
    {
        "model": name,
        "accuracy": accuracy_score(results["label"], results[f"{key}_label"]),
        "macro_f1": f1_score(results["label"], results[f"{key}_label"], labels=LABELS, average="macro"),
        "avg_seconds_per_review": results[f"{key}_seconds"].mean(),
        "cost_per_1000_reviews_usd": 0.0,
    }
    for name, key in MODELS
])
summary.round(4)

## 9. Where each model gets confused

Rows are the real label from the stars. Columns are what the model guessed. A perfect model would only have numbers on the diagonal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (name, key) in zip(axes, MODELS):
    cm = confusion_matrix(results["label"], results[f"{key}_label"], labels=LABELS)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=LABELS, yticklabels=LABELS, ax=ax)
    ax.set_title(name)
    ax.set_xlabel("Model guessed")
    ax.set_ylabel("Real label (from stars)")
plt.tight_layout()
plt.savefig(FIG_DIR / "confusion_matrices.png", dpi=150)
plt.show()

## 10. Scores by star rating

This is the same check as the Kaggle tutorial. If a model works, its score should go up as the stars go up.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=results, x="Score", y="vader_compound", ax=axes[0])
axes[0].set_title("VADER compound score by stars")
sns.barplot(data=results, x="Score", y="roberta_pos", ax=axes[1])
axes[1].set_title("RoBERTa positive score by stars")
plt.tight_layout()
plt.savefig(FIG_DIR / "scores_by_stars.png", dpi=150)
plt.show()

## 11. Where the two models disagree

In [ ]:
disagree = results[results["vader_label"] != results["roberta_label"]]
print(f"The models disagree on {len(disagree)} of {len(results)} reviews")
disagree[["Score", "label", "vader_label", "roberta_label", "Text"]].head(10)

## 12. Try your own review

Change the sentences below and run the cell again.

In [ ]:
def compare(text):
    compound = sia.polarity_scores(text)["compound"]
    probs = roberta_probs(text)
    return {
        "text": text,
        "vader": f"{vader_label(compound)} ({compound:+.2f})",
        "roberta": f"{LABELS[int(probs.argmax())]} ({probs.max():.2f})",
    }


examples = [
    "Best dog treats ever. My puppy goes crazy for them!",
    "The package arrived late but the cookies were fine.",
    "Oh great, another bag of stale chips. Just what I wanted.",
    "Not bad at all, I would buy it again.",
]
pd.DataFrame([compare(t) for t in examples])

## 13. Save results

In [ ]:
results.drop(columns=["Text"]).to_csv(PRED_DIR / "vader_roberta.csv", index=False)
summary.to_csv(PRED_DIR / "summary.csv", index=False)
print("Saved predictions to", PRED_DIR.resolve())
print("Saved charts to", FIG_DIR.resolve())